In [1]:
import mne 
import torch
import numpy as np
import os 
from sklearn.model_selection import train_test_split

In [2]:
subject_ids = range(1,110)
base_path = "E:/EEG_Dataset/eeg-motor-movementimagery-dataset-1.0.0"

In [3]:
# 左右MIが入ってるラン：4 / 8 / 12
RUNS  = ["R04", "R08", "R12"]

# （チャンネルはそのままでもOKだけど、MI寄りには↓が定番）
# PICKS = ["FC3","C3","CP3","FCz","Cz","CPz","FC4","C4"]
# PICKS = ["FC1", "FC2", "C3", "CZ", "C4", "CP1", "CP2", "PZ"]  # ←今使ってるやつ
PICKS = ["FC1", "FC2", "C3", "C4", "CP1", "CP2", "P3", "P4"]


all_x, all_y, all_groups = [], [], []

for subject_id in subject_ids:
    subj = f"S{subject_id:03d}"
    for run in RUNS:
        edf_path = os.path.join(base_path, subj, f"{subj}{run}.edf")
        if not os.path.exists(edf_path):
            continue

        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose="ERROR")
        raw.rename_channels(lambda s: s.replace('.','').upper())
        raw.set_montage('standard_1020', on_missing='ignore')
        raw, _ = mne.set_eeg_reference(raw, ref_channels='average')
        # 事前チェック（無いchがあればこのrunはスキップ）
        missing = [ch for ch in PICKS if ch not in raw.ch_names]
        if missing:
            print(f"[WARN] {subj}{run}: missing channels {missing} → skip")
            continue
        raw.pick_channels(PICKS, ordered=True)
        assert raw.ch_names == PICKS, f"channel order mismatch: {raw.ch_names} != {PICKS}"

        raw.filter(1., 30., fir_design='firwin')
        raw.resample(250)

        events, _ = mne.events_from_annotations(raw)
        event_id = {'T1': 1, 'T2': 2}  # ← このラン群は T1=左, T2=右 

        epochs = mne.Epochs(raw, events, event_id=event_id,
                            tmin=-0.5, tmax=3.5, baseline=(-0.5,0.0),
                            preload=True, verbose="ERROR")

        x = epochs.get_data()
        y_ids = epochs.events[:, 2]
        y = np.where(y_ids == event_id['T1'], 0, 1)  # 左→0, 右→1

        all_x.append(x)
        all_y.append(y)
        all_groups.append(np.full(len(y), subject_id, dtype=int))  # グループ=被験者ID


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 529 samples (3.306 s)

Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
NOTE: pick_channels() is a legacy function. New code should u

In [4]:
fs = 250
tmin, tmax = -0.5, 3.5
expected_T = int(round((tmax - tmin) * fs)) + 1  # = 1001（MNEは両端含む）

all_x_fixed, all_y_fixed, all_groups = [], [], []
for file_idx, (x, y) in enumerate(zip(all_x, all_y)):
    T = x.shape[2]
    if T >= expected_T:
        # 長い分は末尾を切り落として1001に合わせる（≒オフバイワン修正）
        x = x[..., :expected_T]
    else:
         #まれに短い場合だけスキップ or 末尾パディング（必要なら）
        #pad = expected_T - T
        #x = np.pad(x, ((0,0),(0,0),(0,pad)), mode='edge')
        pass
    all_x_fixed.append(x)
    all_y_fixed.append(y)
    all_groups.append(np.full(len(y), file_idx, dtype=int))
x_all = np.concatenate(all_x_fixed, axis=0)
y_all = np.concatenate(all_y_fixed, axis=0)
groups = np.concatenate(all_groups, axis=0)
print("all:", x_all.shape, y_all.shape)



all: (7070, 8, 1001) (7070,)


In [5]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(x_all, y_all, groups))

X_train, X_test = x_all[train_idx], x_all[test_idx]
y_train, y_test = y_all[train_idx], y_all[test_idx]
groups_train = groups[train_idx]

# リーク防止チェック
assert set(groups[train_idx]).isdisjoint(set(groups[test_idx]))
#print("train/test:", X_train.shape, X_test.shape)




In [ ]:
eps = 1e-8
mean_tr = X_train.mean(axis=(0,2), keepdims=True)            # (1,C,1)
std_tr  = X_train.std(axis=(0,2), keepdims=True) + eps

def z(X): return (X - mean_tr) / std_tr

X_train = z(X_train)
X_test  = z(X_test)

# （任意）オンライン用に保存
# np.savez("zscore_stats.npz", mean=mean_tr.squeeze(), std=std_tr.squeeze())


In [ ]:
#学習データのクラスバランス調整
rng = np.random.default_rng(42)
idx0 = np.where(y_train == 0)[0]
idx1 = np.where(y_train == 1)[0]
n = max(len(idx0), len(idx1))
idx0_os = rng.choice(idx0, n, replace=True)
idx1_os = rng.choice(idx1, n, replace=True)
balanced_idx = np.concatenate([idx0_os, idx1_os])
rng.shuffle(balanced_idx)

X_train = X_train[balanced_idx]
y_train = y_train[balanced_idx]
#print("train balanced:", np.bincount(y_train))


train balanced: [3659 3659]


In [7]:
eps = 1e-8
mean_tr = X_train.mean(axis=(0,2), keepdims=True)            # (1,C,1)
std_tr  = X_train.std(axis=(0,2), keepdims=True) + eps

def z(X): return (X - mean_tr) / std_tr

X_train = z(X_train)
X_test  = z(X_test)

# （任意）オンライン用に保存
# np.savez("zscore_stats.npz", mean=mean_tr.squeeze(), std=std_tr.squeeze())

In [8]:
from torch.utils.data import Dataset

class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx]      # (C, T)
        y = self.y[idx]      # ()
        return x, y          # ← y も返す


In [9]:
from torch.utils.data import DataLoader

train_ds = EEGDataset(X_train, y_train)
test_ds  = EEGDataset(X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=128, shuffle=False, num_workers=0, pin_memory=True)

# サニティ
xb, yb = next(iter(train_loader))
#print(xb.shape, yb.shape, xb.dtype, yb.dtype)


In [10]:
import torch.nn as nn
import torch.nn.functional as F

class EEGNet(nn.Module):
    def __init__(self, num_classes=2, channels=8):
        super().__init__()
        self.firstconv = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=(1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(8)
        )
        self.depthwiseConv = nn.Sequential(
            nn.Conv2d(8, 16, kernel_size=(channels, 1), groups=8, bias=False),
            nn.BatchNorm2d(16), nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(0.25)
        )
        self.separableConv = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=(1, 16), padding=(0, 8), bias=False),
            nn.BatchNorm2d(16), nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(0.25)
        )
        self.classify = nn.Linear(16 * 1 * 1, num_classes)  # ← 後でGlobalAvgPoolで 1×1 に

    def forward(self, x):                  # x: (B,C,T)
        x = x.unsqueeze(1)                 # (B,1,C,T)
        x = self.firstconv(x)
        x = self.depthwiseConv(x)
        x = self.separableConv(x)
        x = x.mean(dim=(-1, -2))          # Global Average Pool over (C', T') → (B, 16)
        return self.classify(x)


In [11]:
device = torch.device("cuda")  

C, T = X_train.shape[1], X_train.shape[2]
model = EEGNet(num_classes=2, channels=C).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-3, weight_decay=1e-4)
#scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3)
#scaler = torch.cuda.amp.GradScaler(enabled=(device.type=='cuda'))
scaler = None

In [ ]:
from torch.nn.utils import clip_grad_norm_

def train_one_epoch(model, loader, criterion, optimizer, device, max_grad_norm=None):
    model.train()
    tot_loss, tot_correct, tot = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        if max_grad_norm is not None:
            clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()

        b = y.size(0)
        tot_loss += loss.item() * b
        tot_correct += (logits.argmax(1) == y).sum().item()
        tot += b
    return tot_loss/tot, tot_correct/tot

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    tot_loss, tot_correct, tot = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        b = y.size(0)
        tot_loss += loss.item() * b
        tot_correct += (logits.argmax(1) == y).sum().item()
        tot += b
    return tot_loss/tot, tot_correct/tot


In [14]:
def train(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y).sum().item()
        total += y.size(0)

    acc = correct / total
    return running_loss / len(loader), acc

def evaluate(model, loader, criterion, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == y).sum().item()
            total += y.size(0)

    return correct / total

In [ ]:
epochs = 50
best_test = 0.0

for ep in range(1, epochs+1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    te_loss, te_acc = evaluate(model, test_loader, criterion, device)

    print(f"[{ep:03d}/{epochs}] "
          f"train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | "
          f"test_loss={te_loss:.4f} test_acc={te_acc:.4f}")

    if te_acc > best_test:
        best_test = te_acc
        torch.save(model.state_dict(), "best.pt")

print(f"\nBEST test_acc={best_test:.4f}")


[001/50] train_loss=0.5781 train_acc=0.6909 | test_loss=0.6276 test_acc=0.6523
[002/50] train_loss=0.5739 train_acc=0.6980 | test_loss=0.6516 test_acc=0.6202
[003/50] train_loss=0.5727 train_acc=0.6976 | test_loss=0.6414 test_acc=0.6390
[004/50] train_loss=0.5759 train_acc=0.6995 | test_loss=0.7013 test_acc=0.5700
[005/50] train_loss=0.5781 train_acc=0.6913 | test_loss=0.6392 test_acc=0.6328
[006/50] train_loss=0.5756 train_acc=0.6981 | test_loss=0.6581 test_acc=0.6146
[007/50] train_loss=0.5799 train_acc=0.6906 | test_loss=0.6636 test_acc=0.6244
[008/50] train_loss=0.5711 train_acc=0.7007 | test_loss=0.6674 test_acc=0.6125
[009/50] train_loss=0.5747 train_acc=0.6938 | test_loss=0.6703 test_acc=0.6021
[010/50] train_loss=0.5766 train_acc=0.6953 | test_loss=0.6523 test_acc=0.6174
[011/50] train_loss=0.5728 train_acc=0.6986 | test_loss=0.6700 test_acc=0.6118
[012/50] train_loss=0.5752 train_acc=0.6957 | test_loss=0.6303 test_acc=0.6383
[013/50] train_loss=0.5712 train_acc=0.7024 | test_l

In [16]:
epochs = 50  # 時間と相談して増減してOK

for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
    test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")

Epoch 1/50 - Train Loss: 0.5882, Train Acc: 0.6854, Test Acc: 0.6209
Epoch 2/50 - Train Loss: 0.5872, Train Acc: 0.6882, Test Acc: 0.6334
Epoch 3/50 - Train Loss: 0.5927, Train Acc: 0.6718, Test Acc: 0.6404
Epoch 4/50 - Train Loss: 0.5882, Train Acc: 0.6846, Test Acc: 0.6397
Epoch 5/50 - Train Loss: 0.5899, Train Acc: 0.6823, Test Acc: 0.6098
Epoch 6/50 - Train Loss: 0.5834, Train Acc: 0.6884, Test Acc: 0.5958
Epoch 7/50 - Train Loss: 0.5858, Train Acc: 0.6858, Test Acc: 0.5951
Epoch 8/50 - Train Loss: 0.5873, Train Acc: 0.6842, Test Acc: 0.6056
Epoch 9/50 - Train Loss: 0.5846, Train Acc: 0.6871, Test Acc: 0.6202
Epoch 10/50 - Train Loss: 0.5886, Train Acc: 0.6893, Test Acc: 0.6460
Epoch 11/50 - Train Loss: 0.5826, Train Acc: 0.6905, Test Acc: 0.6404
Epoch 12/50 - Train Loss: 0.5814, Train Acc: 0.6945, Test Acc: 0.6014
Epoch 13/50 - Train Loss: 0.5822, Train Acc: 0.6856, Test Acc: 0.6251
Epoch 14/50 - Train Loss: 0.5835, Train Acc: 0.6868, Test Acc: 0.6077
Epoch 15/50 - Train Loss: 0.5

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,
                          num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False,
                          num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=128, shuffle=False,
                          num_workers=0, pin_memory=False)

# サニティチェック（1バッチだけ取って形と型を確認）
xb, yb = next(iter(train_loader))
print(xb.shape, xb.dtype, yb.shape, yb.dtype)  # 期待: (B,C,T) float32, (B,) int64


In [ ]:
#np.bincount(y_train)